# Certificate test notebook

Local test for the certificate overlay logic used by `POST /api/certificate/send` in `main.py` — same functions, copy-pasted, so what you see here is what production sends.

Run top to bottom. Tune `NAME_Y` / `FONT_SIZE` / `NAME_X` in the config cell until the name lands correctly on `certificate_template.pdf`, then put the same values in `.env` (`CERT_NAME_Y`, `CERT_FONT_SIZE`, `CERT_NAME_X`) and on Vercel.

**Kernel:** use the `myenv` conda environment — it already has `pypdf`, `reportlab`, `arabic-reshaper`, `python-bidi` installed.

In [12]:
import io
import os
import re
from pathlib import Path

from pypdf import PdfReader, PdfWriter
from reportlab.pdfgen import canvas as pdf_canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import arabic_reshaper
from bidi.algorithm import get_display

FONT_NAME = "NotoSansArabic"
pdfmetrics.registerFont(TTFont(FONT_NAME, "NotoSansArabic-Regular.ttf"))

In [13]:
TEMPLATE_PATH = Path("certificate_template.pdf")
OUTDIR = Path("cert_test_output")
OUTDIR.mkdir(exist_ok=True)

NAME_X = None      # None = horizontally centered
NAME_Y = 340        # distance from the bottom of the page, in points
FONT_SIZE = 30

reader = PdfReader(str(TEMPLATE_PATH))
page = reader.pages[0]
PAGE_W, PAGE_H = float(page.mediabox.width), float(page.mediabox.height)
print(f"Template page size: {PAGE_W:.0f} x {PAGE_H:.0f} pt")

Template page size: 842 x 595 pt


In [14]:
def is_arabic(text: str) -> bool:
    return bool(re.search(r'[\u0600-\u06FF]', text))

def make_certificate(name: str, x=None, y=NAME_Y, font_size=FONT_SIZE) -> bytes:
    """Same overlay logic as generate_certificate_bytes() in main.py."""
    if is_arabic(name):
        name = get_display(arabic_reshaper.reshape(name))

    cx = PAGE_W / 2 if x is None else x

    packet = io.BytesIO()
    c = pdf_canvas.Canvas(packet, pagesize=(PAGE_W, PAGE_H))
    c.setFont(FONT_NAME, font_size)
    c.drawCentredString(cx, y, name)
    c.save()
    packet.seek(0)
    overlay = PdfReader(packet)

    template = PdfReader(str(TEMPLATE_PATH))
    tpl_page = template.pages[0]
    tpl_page.merge_page(overlay.pages[0])

    writer = PdfWriter()
    writer.add_page(tpl_page)
    out = io.BytesIO()
    writer.write(out)
    return out.getvalue()

In [15]:
test_name = "Ahmed Alaa"
pdf_bytes = make_certificate(test_name, x=NAME_X, y=NAME_Y, font_size=FONT_SIZE)

out_path = OUTDIR / "preview.pdf"
out_path.write_bytes(pdf_bytes)
print(f"Saved to {out_path.resolve()}")

os.startfile(out_path)  # opens in your default PDF viewer

Saved to C:\Users\ahmed\Downloads\ScholarXBack\mailQR-ScholarX\cert_test_output\preview.pdf


## Iterate on position / size

Generates a few variants side by side so you can compare without editing `NAME_Y`/`FONT_SIZE` over and over.

In [16]:
for y, size in [(250, 26), (300, 30), (340, 34)]:
    pdf_bytes = make_certificate(test_name, y=y, font_size=size)
    path = OUTDIR / f"preview_y{y}_size{size}.pdf"
    path.write_bytes(pdf_bytes)
    print(f"{path.name}  ->  y={y}, font_size={size}")

os.startfile(OUTDIR)  # opens the output folder so you can flip through them

preview_y250_size26.pdf  ->  y=250, font_size=26
preview_y300_size30.pdf  ->  y=300, font_size=30
preview_y340_size34.pdf  ->  y=340, font_size=34


## Arabic name check

Verifies reshaping + right-to-left display work on the actual template.

In [17]:
arabic_test = "أحمد علاء"
pdf_bytes = make_certificate(arabic_test, x=NAME_X, y=NAME_Y, font_size=FONT_SIZE)
out_path = OUTDIR / "preview_arabic.pdf"
out_path.write_bytes(pdf_bytes)
os.startfile(out_path)

## Once you're happy with the values

Copy `NAME_X` / `NAME_Y` / `FONT_SIZE` from the config cell into `.env` as `CERT_NAME_X`, `CERT_NAME_Y`, `CERT_FONT_SIZE` (and mirror them on Vercel). Leave `CERT_NAME_X` unset to keep centering.

---
## Optional: hit the live backend

Only run this if `main.py` is running locally (`uvicorn main:app --reload`) or you point `BACKEND_URL` at your Vercel deployment. Confirms the deployed endpoint renders identically to this notebook.

In [18]:
import requests

BACKEND_URL = "http://localhost:8000"   # or your Vercel URL
ADMIN_KEY = "asaf2026summit"            # from .env

resp = requests.get(
    f"{BACKEND_URL}/api/admin/certificate-preview",
    params={"name": test_name},
    headers={"x-admin-key": ADMIN_KEY},
)
print(resp.status_code)
if resp.ok:
    path = OUTDIR / "preview_from_backend.pdf"
    path.write_bytes(resp.content)
    os.startfile(path)
else:
    print(resp.text)

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/admin/certificate-preview?name=Ahmed+Alaa (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

## Optional: end-to-end send test (real email!)

This actually emails a certificate and marks `cert_sent=True` in the DB — only run it with a real email that (a) is registered and (b) has a `checkins` row for `EVENT_ID`. Uncomment to run.

In [ ]:
CERT_SECRET = "x1dUjFbea6AIePK2eZkpwidwygv28eOb"  # from .env
TEST_EMAIL = "someone-who-attended@example.com"

# resp = requests.post(
#     f"{BACKEND_URL}/api/certificate/send",
#     json={"email": TEST_EMAIL},
#     headers={"x-cert-secret": CERT_SECRET},
# )
# print(resp.status_code, resp.json())